# HydroTwin Final Model Comparison

This notebook evaluates three time series forecasting models:

1. Linear Regression
2. Long Short-Term Memory (LSTM)
3. Gated Recurrent Unit (GRU)

The models predict the next 10-minute vibration reading for two hydropower components:

- Turbine Guide Bearing (TGB)
- Upper Guide Bearing (UGB)

All models use the same 36 historical observations containing four sensor variables.

September validation results are used for model selection. The unseen November 2020 dataset is reserved for final independent evaluation.

RMSE is the primary performance metric, supported by MAE, MAPE, and R squared.

## 1. Import Libraries

The required Python libraries are imported for loading trained models, evaluating forecasting performance, and visualizing the comparison results.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from tensorflow.keras.models import load_model

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

print("Libraries imported successfully.")

## 2. Project Directories

Paths are defined for prepared datasets, saved forecasting models, fitted scalers, and evaluation results.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"

TGB_DIR = PREPARED_DIR / "tgb"
UGB_DIR = PREPARED_DIR / "ugb"

LR_DIR = PROJECT_ROOT / "models" / "linear_regression"
LSTM_DIR = PROJECT_ROOT / "models" / "lstm"
GRU_DIR = PROJECT_ROOT / "models" / "gru"

SCALER_DIR = LSTM_DIR / "scalers"

RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project Root:", PROJECT_ROOT)
print("Results Directory:", RESULTS_DIR)

## 3. Load September Validation Results

The previously saved validation results for Linear Regression, LSTM, and GRU are loaded.

All models were evaluated using the same September validation period. These results are used to determine the preferred forecasting model for each bearing component before final testing.

In [ ]:
lr_validation = pd.read_csv(
    RESULTS_DIR / "linear_regression_validation_results.csv"
)

lstm_validation = pd.read_csv(
    RESULTS_DIR / "lstm_validation_results.csv"
)

gru_validation = pd.read_csv(
    RESULTS_DIR / "gru_validation_results.csv"
)

print("Linear Regression:")
display(lr_validation)

print("LSTM:")
display(lstm_validation)

print("GRU:")
display(gru_validation)

## 4. Three-Model Validation Comparison

The validation results are combined to compare the forecasting performance of Linear Regression, LSTM, and GRU for TGB and UGB.

The comparison focuses on RMSE, MAE, MAPE, and R squared.

In [ ]:
validation_results = pd.concat(
    [
        lr_validation,
        lstm_validation,
        gru_validation
    ],
    ignore_index=True
)

validation_results = validation_results[
    [
        "Component",
        "Model",
        "RMSE",
        "MAE",
        "MAPE",
        "R2"
    ]
]

validation_results = validation_results.sort_values(
    ["Component", "RMSE"]
).reset_index(drop=True)

display(validation_results.round(4))

## 5. Model Selection Based on Validation RMSE

The model achieving the lowest September validation RMSE is selected as the preferred forecasting model for each component.

Model selection is completed before evaluating the unseen November test dataset to prevent test data leakage.

In [ ]:
selected_models = (
    validation_results
    .loc[
        validation_results
        .groupby("Component")["RMSE"]
        .idxmin()
    ]
    .reset_index(drop=True)
)

display(selected_models.round(4))

## 6. Save Validation-Based Model Selection

The best model for each bearing component is selected based on September validation RMSE.

These selections are recorded before the November test evaluation to ensure that the test dataset is not used for model selection or hyperparameter tuning.

In [ ]:
selection_path = (
    RESULTS_DIR / "selected_models_validation.csv"
)

selected_models.to_csv(
    selection_path,
    index=False
)

print("Model selections saved:", selection_path.exists())

display(selected_models.round(4))

## 7. Verify Final Model Files

The six trained forecasting models are verified before the final test evaluation.

Each component has one Linear Regression, one LSTM, and one GRU model.

All models were trained and finalized using training and validation data only.

In [ ]:
model_paths = {
    "TGB Linear Regression":
        LR_DIR / "tgb_linear_regression.pkl",

    "TGB LSTM":
        LSTM_DIR / "tgb_lstm.keras",

    "TGB GRU":
        GRU_DIR / "tgb_gru.keras",

    "UGB Linear Regression":
        LR_DIR / "ugb_linear_regression.pkl",

    "UGB LSTM":
        LSTM_DIR / "ugb_lstm.keras",

    "UGB GRU":
        GRU_DIR / "ugb_gru.keras"
}

for name, path in model_paths.items():
    print(f"{name}: {path.exists()}")

## 8. Verify Feature and Target Scalers

The previously fitted training scalers are verified for both bearing components.

These scalers will be reused to transform November test inputs without fitting any new preprocessing parameters.

In [ ]:
scaler_paths = {
    "TGB Feature Scaler":
        SCALER_DIR / "tgb_feature_scaler.pkl",

    "TGB Target Scaler":
        SCALER_DIR / "tgb_target_scaler.pkl",

    "UGB Feature Scaler":
        SCALER_DIR / "ugb_feature_scaler.pkl",

    "UGB Target Scaler":
        SCALER_DIR / "ugb_target_scaler.pkl"
}

for name, path in scaler_paths.items():
    print(f"{name}: {path.exists()}")

## 9. Load November Test Dataset

The unseen November 2020 dataset is loaded for independent evaluation of the finalized forecasting models.

Each test sequence contains 36 historical observations with four sensor variables.

All models predict vibration at the next 10-minute interval.

No model retraining, hyperparameter tuning, or preprocessing refitting will be performed using this test dataset.

In [ ]:
# TGB test data

X_test_tgb = np.load(
    TGB_DIR / "X_test.npy"
)

y_test_tgb = np.load(
    TGB_DIR / "y_test.npy"
)

test_times_tgb = np.load(
    TGB_DIR / "test_times.npy",
    allow_pickle=True
)

# UGB test data

X_test_ugb = np.load(
    UGB_DIR / "X_test.npy"
)

y_test_ugb = np.load(
    UGB_DIR / "y_test.npy"
)

test_times_ugb = np.load(
    UGB_DIR / "test_times.npy",
    allow_pickle=True
)

print("November test data loaded successfully.")

## 10. Verify Test Dataset Integrity

The dimensions, target lengths, and timestamps of both bearing datasets are checked before generating model predictions.

The TGB and UGB test datasets must contain the same number of observations with aligned target timestamps.

In [ ]:
print("TGB Test Dataset")
print("Input shape:", X_test_tgb.shape)
print("Target shape:", y_test_tgb.shape)

print()

print("UGB Test Dataset")
print("Input shape:", X_test_ugb.shape)
print("Target shape:", y_test_ugb.shape)

print()

print(
    "Timestamps aligned:",
    np.array_equal(
        test_times_tgb,
        test_times_ugb
    )
)

print(
    "TGB values finite:",
    np.isfinite(X_test_tgb).all()
    and np.isfinite(y_test_tgb).all()
)

print(
    "UGB values finite:",
    np.isfinite(X_test_ugb).all()
    and np.isfinite(y_test_ugb).all()
)

## 11. Load Finalized Forecasting Models

The previously trained Linear Regression, LSTM, and GRU models are loaded for both TGB and UGB.

All models were finalized using training and validation data before accessing the November test dataset.

No additional training or hyperparameter tuning is performed during the final evaluation.

In [ ]:
# Load Linear Regression models

tgb_lr_model = joblib.load(
    LR_DIR / "tgb_linear_regression.pkl"
)

ugb_lr_model = joblib.load(
    LR_DIR / "ugb_linear_regression.pkl"
)

# Load LSTM models

tgb_lstm_model = load_model(
    LSTM_DIR / "tgb_lstm.keras",
    compile=False
)

ugb_lstm_model = load_model(
    LSTM_DIR / "ugb_lstm.keras",
    compile=False
)

# Load GRU models

tgb_gru_model = load_model(
    GRU_DIR / "tgb_gru.keras",
    compile=False
)

ugb_gru_model = load_model(
    GRU_DIR / "ugb_gru.keras",
    compile=False
)

print("All six models loaded successfully.")

## 12. Load Training Feature and Target Scalers

The feature and target scalers previously fitted using the training dataset are loaded.

LSTM and GRU share the same component-specific scalers to ensure consistent preprocessing.

The scalers are not refitted on the November test dataset.

In [ ]:
# TGB scalers

tgb_feature_scaler = joblib.load(
    SCALER_DIR / "tgb_feature_scaler.pkl"
)

tgb_target_scaler = joblib.load(
    SCALER_DIR / "tgb_target_scaler.pkl"
)

# UGB scalers

ugb_feature_scaler = joblib.load(
    SCALER_DIR / "ugb_feature_scaler.pkl"
)

ugb_target_scaler = joblib.load(
    SCALER_DIR / "ugb_target_scaler.pkl"
)

print("All four scalers loaded successfully.")

## 13. Prepare Test Inputs for Each Model

The November test sequences are prepared according to the input requirements of each forecasting model.

Linear Regression uses flattened historical observations, while LSTM and GRU use scaled three-dimensional sequences.

All three models receive information from the same 36 historical observations and predict the same next-step vibration target.

In [ ]:
# Flatten test data for Linear Regression

X_test_tgb_flat = X_test_tgb.reshape(
    X_test_tgb.shape[0],
    -1
)

X_test_ugb_flat = X_test_ugb.reshape(
    X_test_ugb.shape[0],
    -1
)

# Scale TGB inputs for LSTM and GRU

X_test_tgb_scaled = (
    tgb_feature_scaler
    .transform(
        X_test_tgb.reshape(-1, 4)
    )
    .reshape(X_test_tgb.shape)
)

# Scale UGB inputs for LSTM and GRU

X_test_ugb_scaled = (
    ugb_feature_scaler
    .transform(
        X_test_ugb.reshape(-1, 4)
    )
    .reshape(X_test_ugb.shape)
)

print("TGB flattened:", X_test_tgb_flat.shape)
print("TGB scaled:", X_test_tgb_scaled.shape)

print()

print("UGB flattened:", X_test_ugb_flat.shape)
print("UGB scaled:", X_test_ugb_scaled.shape)

## 14. TGB Final Test Predictions

The finalized Linear Regression, LSTM, and GRU models generate one-step-ahead vibration predictions for the TGB component.

LSTM and GRU predictions are inverse-transformed to the original vibration scale to allow a fair comparison with Linear Regression.

All predictions correspond to the same November test timestamps.

In [ ]:
# Linear Regression predictions

tgb_lr_pred = tgb_lr_model.predict(
    X_test_tgb_flat
).ravel()

# LSTM predictions

tgb_lstm_pred_scaled = tgb_lstm_model.predict(
    X_test_tgb_scaled,
    verbose=0
)

tgb_lstm_pred = (
    tgb_target_scaler
    .inverse_transform(
        tgb_lstm_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

# GRU predictions

tgb_gru_pred_scaled = tgb_gru_model.predict(
    X_test_tgb_scaled,
    verbose=0
)

tgb_gru_pred = (
    tgb_target_scaler
    .inverse_transform(
        tgb_gru_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

print("TGB predictions generated successfully.")

## 15. UGB Final Test Predictions

The finalized Linear Regression, LSTM, and GRU models generate one-step-ahead vibration predictions for the UGB component.

The previously fitted UGB target scaler is used to inverse-transform recurrent neural network predictions into the original vibration scale.

All models are evaluated on the same November test observations.

In [ ]:
# Linear Regression predictions

ugb_lr_pred = ugb_lr_model.predict(
    X_test_ugb_flat
).ravel()

# LSTM predictions

ugb_lstm_pred_scaled = ugb_lstm_model.predict(
    X_test_ugb_scaled,
    verbose=0
)

ugb_lstm_pred = (
    ugb_target_scaler
    .inverse_transform(
        ugb_lstm_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

# GRU predictions

ugb_gru_pred_scaled = ugb_gru_model.predict(
    X_test_ugb_scaled,
    verbose=0
)

ugb_gru_pred = (
    ugb_target_scaler
    .inverse_transform(
        ugb_gru_pred_scaled.reshape(-1, 1)
    )
    .ravel()
)

print("UGB predictions generated successfully.")

## 16. Verify Final Test Predictions

The six model predictions are checked to confirm that their lengths match the corresponding test targets.

All prediction values must be finite before performing the final performance evaluation.

In [ ]:
predictions = {
    "TGB Linear Regression": tgb_lr_pred,
    "TGB LSTM": tgb_lstm_pred,
    "TGB GRU": tgb_gru_pred,
    "UGB Linear Regression": ugb_lr_pred,
    "UGB LSTM": ugb_lstm_pred,
    "UGB GRU": ugb_gru_pred
}

for name, pred in predictions.items():

    print(
        f"{name}: "
        f"Shape = {pred.shape}, "
        f"Finite = {np.isfinite(pred).all()}"
    )

assert all(
    pred.shape == (3919,) and np.isfinite(pred).all()
    for pred in predictions.values()
)

print("\nAll predictions verified successfully.")

## 17. Final Test Performance Evaluation

The finalized forecasting models are evaluated using the unseen November 2020 test dataset.

Four performance metrics are calculated:

- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- Mean Absolute Percentage Error (MAPE)
- R squared (R²)

RMSE is the primary evaluation metric. All predictions are evaluated in their original vibration measurement scale.

No model parameters are modified based on the test results.

In [ ]:
def evaluate_model(component, model_name, y_true, y_pred):

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(
        y_true, y_pred
    )

    mape = (
        mean_absolute_percentage_error(
            y_true, y_pred
        ) * 100
    )

    r2 = r2_score(
        y_true, y_pred
    )

    return {
        "Component": component,
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }


print("Evaluation function created successfully.")

## 18. Final Model Comparison on November Test Data

The forecasting performance of Linear Regression, LSTM, and GRU is compared for TGB and UGB.

All three models are evaluated using identical test target observations for each component.

The results provide an independent assessment of model generalization beyond the training and validation periods.

In [ ]:
final_results = pd.DataFrame([

    # TGB Models

    evaluate_model(
        "TGB",
        "Linear Regression",
        y_test_tgb,
        tgb_lr_pred
    ),

    evaluate_model(
        "TGB",
        "LSTM",
        y_test_tgb,
        tgb_lstm_pred
    ),

    evaluate_model(
        "TGB",
        "GRU",
        y_test_tgb,
        tgb_gru_pred
    ),

    # UGB Models

    evaluate_model(
        "UGB",
        "Linear Regression",
        y_test_ugb,
        ugb_lr_pred
    ),

    evaluate_model(
        "UGB",
        "LSTM",
        y_test_ugb,
        ugb_lstm_pred
    ),

    evaluate_model(
        "UGB",
        "GRU",
        y_test_ugb,
        ugb_gru_pred
    )

])

display(final_results.round(4))

## 19. Final Test Performance Ranking

The final test results are ranked separately for TGB and UGB according to RMSE.

Lower RMSE indicates better forecasting accuracy on the November test dataset.

The ranking is used to describe observed test performance. The validation-based model selections remain unchanged.

In [ ]:
ranked_results = (
    final_results
    .sort_values(
        by=["Component", "RMSE"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

ranked_results["Rank"] = (
    ranked_results
    .groupby("Component")
    .cumcount() + 1
)

display(
    ranked_results[
        [
            "Component",
            "Rank",
            "Model",
            "RMSE",
            "MAE",
            "MAPE",
            "R2"
        ]
    ].round(4)
)

## 20. Save Final Model Comparison Results

The November test performance metrics and model rankings are saved as CSV files.

These results will support the evaluation chapter of the HydroTwin project and the subsequent digital twin dashboard development.

In [ ]:
final_results_path = (
    RESULTS_DIR / "final_model_comparison.csv"
)

ranked_results_path = (
    RESULTS_DIR / "final_model_ranking.csv"
)

final_results.to_csv(
    final_results_path,
    index=False
)

ranked_results.to_csv(
    ranked_results_path,
    index=False
)

print("Final comparison saved:", final_results_path.exists())

print("Final ranking saved:", ranked_results_path.exists())

## 21. Test Distribution Analysis

The vibration distributions of September validation data and November test data are compared to investigate changes in component behaviour.

This analysis is descriptive only. The finalized models are not retrained, reselected, or tuned using test observations.

In [ ]:
# Load existing September validation targets
y_val_tgb = np.load(TGB_DIR / "y_val.npy")
y_val_ugb = np.load(UGB_DIR / "y_val.npy")

distribution_check = pd.DataFrame([
    {
        "Component": component,
        "Dataset": period,
        "Mean": values.mean(),
        "Std": values.std(),
        "Min": values.min(),
        "Max": values.max()
    }
    for component, period, values in [
        ("TGB", "September Validation", y_val_tgb),
        ("TGB", "November Test", y_test_tgb),
        ("UGB", "September Validation", y_val_ugb),
        ("UGB", "November Test", y_test_ugb)
    ]
])

display(distribution_check.round(4))

## 22. UGB Generalization Analysis

The November UGB test dataset contains a wider vibration range than the training and validation datasets.

This analysis investigates model performance when actual vibration readings fall outside the range observed during training.

The analysis is conducted only to explain final test performance. No models are retrained or tuned.

In [ ]:
# Load UGB training targets for reference

y_train_ugb = np.load(
    UGB_DIR / "y_train.npy"
)

train_min = y_train_ugb.min()
train_max = y_train_ugb.max()

print("UGB Training Minimum:", train_min)
print("UGB Training Maximum:", train_max)

print("\nNovember Test Range:")
print("Minimum:", y_test_ugb.min())
print("Maximum:", y_test_ugb.max())

# Identify November readings outside training range

below_training = y_test_ugb < train_min
within_training = (
    (y_test_ugb >= train_min) &
    (y_test_ugb <= train_max)
)

print("\nNovember observations:")
print("Below training minimum:", below_training.sum())
print("Within training range:", within_training.sum())

print(
    "Percentage below training minimum:",
    f"{below_training.mean() * 100:.2f}%"
)

## 23. UGB Prediction Range and Error Analysis

The prediction ranges and forecasting errors of Linear Regression, LSTM, and GRU are compared.

Performance is examined separately for test observations below the training target range and within the training target range.

These results support interpretation of model generalization under changing vibration distributions.

In [ ]:
ugb_predictions = {
    "Linear Regression": ugb_lr_pred,
    "LSTM": ugb_lstm_pred,
    "GRU": ugb_gru_pred
}

diagnostic_rows = []

for model_name, pred in ugb_predictions.items():

    for group_name, mask in [
        ("Below Training Range", below_training),
        ("Within Training Range", within_training)
    ]:

        if mask.sum() == 0:
            continue

        diagnostic_rows.append({
            "Model": model_name,
            "Observation Group": group_name,
            "Samples": int(mask.sum()),

            "Actual Mean": y_test_ugb[mask].mean(),
            "Predicted Mean": pred[mask].mean(),

            "RMSE": np.sqrt(
                mean_squared_error(
                    y_test_ugb[mask],
                    pred[mask]
                )
            ),

            "MAE": mean_absolute_error(
                y_test_ugb[mask],
                pred[mask]
            )
        })

ugb_diagnostic_results = pd.DataFrame(
    diagnostic_rows
)

display(ugb_diagnostic_results.round(4))

## 24. Save UGB Generalization Analysis

The diagnostic analysis is saved to document how the forecasting models perform when November vibration readings fall outside the training target range.

These findings are used to interpret the final test results rather than to modify the trained models.

In [ ]:
diagnostic_path = (
    RESULTS_DIR / "ugb_generalization_analysis.csv"
)

ugb_diagnostic_results.to_csv(
    diagnostic_path,
    index=False
)

print(
    "UGB diagnostic results saved:",
    diagnostic_path.exists()
)

## 25. Final Test RMSE Comparison

The November test RMSE values are visualized separately for TGB and UGB.

The comparison illustrates the forecasting performance of Linear Regression, LSTM, and GRU on unseen observations.

Lower RMSE represents better prediction accuracy.

In [ ]:
for component in ["TGB", "UGB"]:

    component_results = final_results[
        final_results["Component"] == component
    ]

    plt.figure(figsize=(8, 5))

    bars = plt.bar(
        component_results["Model"],
        component_results["RMSE"]
    )

    for bar in bars:
        height = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f"{height:.4f}",
            ha="center"
        )

    plt.title(
        f"{component} Final Test RMSE Comparison"
    )

    plt.xlabel("Forecasting Model")
    plt.ylabel("RMSE")
    plt.ylim(
        0,
        component_results["RMSE"].max() * 1.2
    )

    plt.tight_layout()

    plt.savefig(
        RESULTS_DIR /
        f"{component.lower()}_test_rmse_comparison.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## 26. UGB Actual vs Predicted Vibration Under Distribution Shift

A selected interval from the November test dataset is visualized to compare actual UGB vibration with predictions from all three models.

The interval is centered around an observation below the training target range.

The visualization helps explain differences in generalization performance when the test vibration distribution changes.

In [ ]:
# Find a test observation below the training range

below_indices = np.where(below_training)[0]

focus_index = below_indices[
    len(below_indices) // 2
]

# Select a window around that observation

start = max(0, focus_index - 80)
end = min(len(y_test_ugb), focus_index + 80)

x_axis = np.arange(start, end)

plt.figure(figsize=(14, 5))

plt.plot(
    x_axis,
    y_test_ugb[start:end],
    label="Actual UGB",
    linewidth=2
)

plt.plot(
    x_axis,
    ugb_lr_pred[start:end],
    label="Linear Regression"
)

plt.plot(
    x_axis,
    ugb_lstm_pred[start:end],
    label="LSTM"
)

plt.plot(
    x_axis,
    ugb_gru_pred[start:end],
    label="GRU"
)

plt.axhline(
    y=train_min,
    linestyle="--",
    label="Training Minimum"
)

plt.xlabel("November Test Observation")
plt.ylabel("UGB Vibration")

plt.title(
    "UGB Actual vs Predicted Vibration "
    "Around a Below-Training-Range Observation"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "ugb_distribution_shift_predictions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 27. Save Timestamp-Aligned Model Predictions

The actual vibration measurements and predictions from Linear Regression, LSTM, and GRU are saved together with their corresponding November timestamps.

Separate files are created for TGB and UGB to support historical replay, digital twin visualization, and model performance analysis.

In [ ]:
# TGB prediction records

tgb_prediction_results = pd.DataFrame({
    "timestamp": pd.to_datetime(test_times_tgb),

    "actual_vibration": y_test_tgb,

    "linear_regression_prediction": tgb_lr_pred,

    "lstm_prediction": tgb_lstm_pred,

    "gru_prediction": tgb_gru_pred
})

# UGB prediction records

ugb_prediction_results = pd.DataFrame({
    "timestamp": pd.to_datetime(test_times_ugb),

    "actual_vibration": y_test_ugb,

    "linear_regression_prediction": ugb_lr_pred,

    "lstm_prediction": ugb_lstm_pred,

    "gru_prediction": ugb_gru_pred
})

# Save files

tgb_prediction_results.to_csv(
    RESULTS_DIR / "tgb_final_predictions.csv",
    index=False
)

ugb_prediction_results.to_csv(
    RESULTS_DIR / "ugb_final_predictions.csv",
    index=False
)

print("TGB prediction records:", len(tgb_prediction_results))
print("UGB prediction records:", len(ugb_prediction_results))

print(
    "TGB file saved:",
    (RESULTS_DIR / "tgb_final_predictions.csv").exists()
)

print(
    "UGB file saved:",
    (RESULTS_DIR / "ugb_final_predictions.csv").exists()
)

## 28. Conclusion

Three forecasting models—Linear Regression, LSTM, and GRU—were evaluated for next-step vibration prediction of the Turbine Guide Bearing (TGB) and Upper Guide Bearing (UGB).

The models used 36 historical observations to predict vibration at the next 10-minute interval.

On the independent November 2020 test dataset, LSTM achieved the lowest RMSE for TGB (0.6611), while Linear Regression achieved the lowest RMSE for UGB (0.6077).

A distribution-shift analysis revealed that 43.05% of the November UGB observations were below the minimum vibration value observed in the training targets. The recurrent models exhibited larger prediction errors in this region, contributing to their weaker generalization.

The original September validation-based selections remain GRU for TGB and LSTM for UGB. The November rankings are reported separately as independent test findings.

All trained models, evaluation metrics, and timestamp-aligned predictions are preserved for the HydroTwin digital twin prototype.

The next stage develops a historical data-driven digital twin that represents both bearing components and supports synchronized monitoring, forecasting, and visualization.